# Mission Expérimentale : Fine-Tuning Médical (QLoRA)
Ce notebook a pour but d'entraîner le modèle `Phi-3.5-mini` sur le dataset médical `ruslanmv/ai-medical-chatbot` afin de créer un assistant spécialisé en santé.
**Instructions :** Exécutez toutes les cellules de haut en bas sur Google Colab (avec un GPU T4 activé).

In [ ]:
# 1. Installation des dépendances
!pip install -q -U transformers datasets bitsandbytes accelerate peft

In [ ]:
import torch
import re
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Paramètres principaux
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
DATASET_NAME = "ruslanmv/ai-medical-chatbot"
NEW_MODEL = "Phi-3.5-Medical-Adapter"

## 2. Mission Data : Téléchargement, Anonymisation et Tokenization du Dataset

In [ ]:
# Chargement du dataset depuis HuggingFace
raw_dataset = load_dataset(DATASET_NAME, split="train")

# Fonction de nettoyage et anonymisation de sécurité (RGPD / HIPAA)
def anonymize_text(text):
    if not text: return text
    text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '[EMAIL PROTECTED]', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE REDACTED]', text)
    return text

# Formatage pour l'entraînement (Prompt Engineering)
def to_text(example):
    patient_text = anonymize_text(example['Patient'])
    doctor_text = anonymize_text(example['Doctor'])
    text = f"<|user|>\n{patient_text}<|end|>\n<|assistant|>\n{doctor_text}<|end|>"
    return {"text": text}

texts = [to_text(x) for x in raw_dataset]
hf_dataset = Dataset.from_list(texts)

# Chargement du Tokenizer pour préparer les données
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Application de la tokenization
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Pour l'exercice (temps limité sur Colab), on ne prend qu'un sous-échantillon (ex: 2000 lignes)
train_dataset = tokenized_dataset.shuffle(seed=42).select(range(2000))
print(f"Taille du dataset d'entraînement prêt : {len(train_dataset)}")

## 3. Chargement du Modèle en 4-bit (QLoRA) et Configuration LoRA

In [ ]:
# Configuration de la quantification 4-bit pour économiser la RAM GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Chargement du modèle de base
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.use_cache = False

# Préparation du modèle pour LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Entraînement

In [ ]:
# Paramètres d'entraînement robustes
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1, # 1 seule epoch pour la démonstration
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    fp16=True,
    report_to="none",
    max_steps=100, # Limite à 100 itérations pour le POC
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Initialisation du Trainer classique (sans TRL)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

# Lancement de l'entraînement
trainer.train()

## 5. Sauvegarde de l'Adaptateur

In [ ]:
# Sauvegarder les poids LoRA
trainer.model.save_pretrained(NEW_MODEL)
print(f"Modèle sauvegardé dans le dossier : {NEW_MODEL}")